In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pip install transformers datasets rouge-score

In [ ]:
from datasets import load_dataset
dataset = load_dataset('cnn_dailymail','3.0.0')

In [ ]:
  print("\nSample article:")
print(dataset["train"][0]["article"])

print("\nReference summary:")
print(dataset["train"][0]["highlights"])

In [ ]:
small_dataset = dataset['train'].select(range(1000))
print(f"Subset Size {len(small_dataset)}")

In [ ]:
print(dataset)

In [ ]:
from transformers import AutoTokenizer
import nltk
import re
nltk.download('punkt')

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[[^]]*\]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = text.strip()
    return text

def preprocess_dataset(dataset, num_samples=1000):
    cleaned_articles = []
    cleaned_summaries = []

    for i in range(num_samples):
        article = clean_text(dataset['train'][i]['article'])
        summary = clean_text(dataset['train'][i]['highlights'])
        cleaned_articles.append(article)
        cleaned_summaries.append(summary)

    return cleaned_articles, cleaned_summaries

articles, summaries = preprocess_dataset(dataset, 1000)
print("✅ Preprocessing complete. Example:")
print("Article:", articles[0][:300])
print("\nSummary:", summaries[0])

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print("BART model and tokenizer loaded successfully!")


In [ ]:
inputs = tokenizer(
    articles,
    max_length=1024,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

labels = tokenizer(
    summaries,
    max_length=150,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

print("✅ Tokenization complete.")
print("Input tensor shape:", inputs["input_ids"].shape)
print("Label tensor shape:", labels["input_ids"].shape)


In [ ]:
tokenized = small_dataset.map(preprocessing,batched=True)
print(tokenized.column_names )

In [ ]:
print(tokenized[0].keys())
print("\nInput IDs length:", len(tokenized[0]["input_ids"]))
print("\nLabel IDs length:", len(tokenized[0]["labels"]))